In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

In [10]:
df_base = pd.read_parquet('../../data/gold/forecasting/ds_base_dataset.parquet')[['week_date','time_series_id','units_sold']]
df_segmented = pd.read_parquet('../../data/gold/forecasting/ds_segmented_time_series.parquet')

In [11]:
df_base = df_base.merge(df_segmented,on='time_series_id', how='left')
df_base = df_base[df_base['segment']=='valid']

In [12]:
def detect_outliers_rolling_median(ts_data, window=13, cutoff_multiplier=3.0, method='median'):
    """
    Detecta outliers usando mediana móvel + MAD escalado.

    Parameters
    ----------
    ts_data : np.ndarray (float)  ← must be a plain numpy float array
    window : int       Janela centrada (semanas)
    cutoff_multiplier  Multiplicador de MAD (3.0 = balanceado)
    method : 'median' ou 'min' (P10 robusto)

    Returns
    -------
    outlier_mask, rolling_baseline, upper_threshold, lower_threshold
    """
    n = len(ts_data)
    outlier_mask = np.zeros(n, dtype=bool)

    non_zero_mask = ts_data > 0
    if non_zero_mask.sum() < window:
        nan = np.full(n, np.nan)
        return outlier_mask, nan, nan, nan

    rolling_baseline = np.full(n, np.nan)
    rolling_mad      = np.full(n, np.nan)

    for i in range(n):
        start = max(0, i - window // 2)
        end   = min(n, i + window // 2 + 1)
        window_non_zero = ts_data[start:end]
        window_non_zero = window_non_zero[window_non_zero > 0]

        if len(window_non_zero) < 3:
            continue

        baseline = np.median(window_non_zero) if method != 'min' else np.percentile(window_non_zero, 10)
        rolling_baseline[i] = baseline
        rolling_mad[i]      = np.median(np.abs(window_non_zero - baseline))

    rolling_std = 1.4826 * rolling_mad
    upper_threshold = rolling_baseline + cutoff_multiplier * rolling_std
    lower_threshold = np.maximum(rolling_baseline - cutoff_multiplier * rolling_std, 0)

    for i in range(n):
        if ts_data[i] > 0 and not np.isnan(upper_threshold[i]):
            if ts_data[i] > upper_threshold[i] or ts_data[i] < lower_threshold[i]:
                outlier_mask[i] = True

    return outlier_mask, rolling_baseline, upper_threshold, lower_threshold


def apply_rolling_median_detection(df_base, window=13, cutoff_multiplier=3.0, method='median'):
    """Aplica detecção de outliers em todas as séries."""

    print("🔍 DETECÇÃO DE OUTLIERS - MEDIANA MÓVEL")
    print("=" * 70)
    print(f"   Janela: {window} semanas")
    print(f"   Cutoff: {cutoff_multiplier}×MAD")
    print(f"   Método: {method}")

    df = df_base.copy()
    df['is_outlier']       = False
    df['rolling_baseline'] = np.nan
    df['upper_threshold']  = np.nan
    df['lower_threshold']  = np.nan

    total_outliers       = 0
    series_with_outliers = 0

    for ts_id in df['time_series_id'].unique():
        mask    = df['time_series_id'] == ts_id
        ts_data = df.loc[mask].sort_values('week_date')

        # to_numpy(float) avoids IntegerArray issues from nullable Int64 parquet columns
        units = ts_data['units_sold'].to_numpy(dtype=float, na_value=0.0)

        outlier_mask, baseline, upper, lower = detect_outliers_rolling_median(
            units, window=window, cutoff_multiplier=cutoff_multiplier, method=method
        )

        df.loc[ts_data.index, 'is_outlier']       = outlier_mask
        df.loc[ts_data.index, 'rolling_baseline'] = baseline
        df.loc[ts_data.index, 'upper_threshold']  = upper
        df.loc[ts_data.index, 'lower_threshold']  = lower

        n_outliers = outlier_mask.sum()
        if n_outliers > 0:
            series_with_outliers += 1
            total_outliers       += n_outliers

    print(f"\n📊 RESULTADOS:")
    print(f"   Séries com outliers: {series_with_outliers:,} / {df['time_series_id'].nunique():,}")
    print(f"   Total de outliers: {total_outliers:,} ({total_outliers / len(df):.2%} dos registros)")

    print(f"\n🔝 Top 10 séries com mais outliers:")
    outlier_counts = df[df['is_outlier']].groupby('time_series_id').size().sort_values(ascending=False).head(10)
    for ts_id, count in outlier_counts.items():
        total_weeks = (df['time_series_id'] == ts_id).sum()
        print(f"      Série {ts_id}: {count} outliers ({count / total_weeks:.1%} das semanas)")

    return df


def treat_outliers_rolling_median(df_with_outliers):
    """
    Cria 4 versões do target:
      units_sold_original     — sem alteração
      units_sold_capped       — outliers limitados ao upper_threshold
      units_sold_removed      — outliers substituídos pela rolling_baseline
      units_sold_interpolated — outliers marcados como NaN e interpolados
                                linearmente entre os vizinhos válidos
    """

    print("\n🔧 TRATAMENTO DE OUTLIERS")
    print("=" * 70)

    df = df_with_outliers.copy()

    # Cast to float64 so we can safely assign float thresholds/baselines
    # (units_sold is Int64 from parquet; direct assignment of floats would raise TypeError)
    df['units_sold_original']     = df['units_sold'].astype('float64')
    df['units_sold_capped']       = df['units_sold'].astype('float64')
    df['units_sold_removed']      = df['units_sold'].astype('float64')
    df['units_sold_interpolated'] = df['units_sold'].astype('float64')

    # Capped: clip to upper threshold
    capped_mask = df['is_outlier'] & (df['units_sold_capped'] > df['upper_threshold'])
    df.loc[capped_mask, 'units_sold_capped'] = df.loc[capped_mask, 'upper_threshold']

    # Removed: replace with rolling baseline
    df.loc[df['is_outlier'], 'units_sold_removed'] = df.loc[df['is_outlier'], 'rolling_baseline']

    # Interpolated: set outliers to NaN, then interpolate per series
    df.loc[df['is_outlier'], 'units_sold_interpolated'] = np.nan
    df['units_sold_interpolated'] = (
        df.groupby('time_series_id', group_keys=False)['units_sold_interpolated']
        .apply(lambda s: s.interpolate(method='linear', limit_direction='both'))
    )
    # Any remaining NaN (series where all points are outliers) falls back to 0
    df['units_sold_interpolated'] = df['units_sold_interpolated'].fillna(0)

    print(f"   Outliers capped:       {capped_mask.sum():,}")
    print(f"   Outliers removed:      {df['is_outlier'].sum():,}")
    print(f"   Outliers interpolated: {df['is_outlier'].sum():,}")

    return df


# =========================================================================
# PIPELINE
# =========================================================================

df_detected = apply_rolling_median_detection(
    df_base,
    window=13,
    cutoff_multiplier=3.0,
    method='median'
)

df_treated = treat_outliers_rolling_median(df_detected)

output_path = '../../data/gold/forecasting/ds_time_series_outliers_detected.parquet'
df_treated.to_parquet(output_path)
print(f"\n💾 Salvo em: {output_path}")

🔍 DETECÇÃO DE OUTLIERS - MEDIANA MÓVEL
   Janela: 13 semanas
   Cutoff: 3.0×MAD
   Método: median

📊 RESULTADOS:
   Séries com outliers: 1,889 / 2,232
   Total de outliers: 12,682 (6.82% dos registros)

🔝 Top 10 séries com mais outliers:
      Série 1041: 27 outliers (26.0% das semanas)
      Série 1347: 26 outliers (25.0% das semanas)
      Série 755: 25 outliers (24.0% das semanas)
      Série 575: 22 outliers (21.2% das semanas)
      Série 1277: 22 outliers (21.2% das semanas)
      Série 388: 22 outliers (21.2% das semanas)
      Série 597: 20 outliers (19.2% das semanas)
      Série 455: 20 outliers (19.2% das semanas)
      Série 1496: 20 outliers (19.2% das semanas)
      Série 386: 20 outliers (19.2% das semanas)

🔧 TRATAMENTO DE OUTLIERS
   Outliers capped:       11,574
   Outliers removed:      12,682
   Outliers interpolated: 12,682

💾 Salvo em: ../../data/gold/forecasting/ds_time_series_outliers_detected.parquet


In [13]:
import os

PLOT_DIR = '../../plots/outliers/'
os.makedirs(PLOT_DIR, exist_ok=True)


def plot_series_before_after(df_treated, ts_id, save_dir=PLOT_DIR):
    """
    2-panel before/after plot for one time series.
    Top panel  : original + rolling baseline + thresholds + outlier markers
    Bottom panel: original vs capped vs removed vs interpolated
    """
    ts = df_treated[df_treated['time_series_id'] == ts_id].sort_values('week_date')

    dates        = ts['week_date'].values
    original     = ts['units_sold_original'].values
    capped       = ts['units_sold_capped'].values
    removed      = ts['units_sold_removed'].values
    interpolated = ts['units_sold_interpolated'].values
    baseline     = ts['rolling_baseline'].values
    upper        = ts['upper_threshold'].values
    lower        = ts['lower_threshold'].values
    is_out       = ts['is_outlier'].values

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

    # --- Panel 1: Detection ---
    ax1.plot(dates, original, 'o-', color='steelblue', lw=1.5, ms=3,
             label='Observado', alpha=0.7)
    ax1.plot(dates, baseline, '--', color='green', lw=2,
             label='Baseline (mediana móvel)', alpha=0.8)
    ax1.plot(dates, upper, ':', color='red', lw=1.5,
             label='Upper threshold', alpha=0.7)
    ax1.plot(dates, lower, ':', color='orange', lw=1.5,
             label='Lower threshold', alpha=0.7)
    ax1.fill_between(dates, lower, upper, color='green', alpha=0.08)

    if is_out.sum() > 0:
        ax1.scatter(dates[is_out], original[is_out],
                    color='red', s=120, marker='X', zorder=5,
                    label=f'Outliers ({is_out.sum()})')

    ax1.set_title(f'Série {ts_id} — Detecção', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Unidades vendidas')
    ax1.legend(fontsize=9, loc='upper left')
    ax1.grid(alpha=0.3)

    # --- Panel 2: Comparison ---
    ax2.plot(dates, original, 'o-', color='steelblue', lw=1.5, ms=3,
             label='Original', alpha=0.4)
    ax2.plot(dates, capped, 's-', color='darkorange', lw=1.5, ms=3,
             label='Capped', alpha=0.8)
    #ax2.plot(dates, removed, '^-', color='green', lw=1.5, ms=3,
    #         label='Removed (baseline)', alpha=0.8)
    #ax2.plot(dates, interpolated, 'D-', color='purple', lw=1.5, ms=3,
    #         label='Interpolated', alpha=0.8)

    if is_out.sum() > 0:
        ax2.scatter(dates[is_out], original[is_out],
                    color='red', s=60, zorder=5,
                    label=f'Outliers originais ({is_out.sum()})')

    ax2.set_title(f'Série {ts_id} — Antes vs Depois', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Data')
    ax2.set_ylabel('Unidades vendidas')
    ax2.legend(fontsize=9, loc='upper left')
    ax2.grid(alpha=0.3)
    ax2.tick_params(axis='x', rotation=30)

    plt.tight_layout()
    path = os.path.join(save_dir, f'serie_{ts_id}.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    return path


def generate_before_after_plots(df_treated, n=20, save_dir=PLOT_DIR):
    """Gera plots das N séries com mais outliers."""
    top_series = (
        df_treated[df_treated['is_outlier']]
        .groupby('time_series_id').size()
        .nlargest(n).index.tolist()
    )

    print(f"📊 Gerando {len(top_series)} plots em '{save_dir}' ...")
    for i, ts_id in enumerate(top_series, 1):
        path = plot_series_before_after(df_treated, ts_id, save_dir)
        print(f"   [{i:>2}/{len(top_series)}] {path}")

    print(f"\n✅ Plots salvos em: {save_dir}")


generate_before_after_plots(df_treated, n=20)

📊 Gerando 20 plots em '../../plots/outliers/' ...
   [ 1/20] ../../plots/outliers/serie_1041.png
   [ 2/20] ../../plots/outliers/serie_1347.png
   [ 3/20] ../../plots/outliers/serie_755.png
   [ 4/20] ../../plots/outliers/serie_388.png
   [ 5/20] ../../plots/outliers/serie_575.png
   [ 6/20] ../../plots/outliers/serie_1277.png
   [ 7/20] ../../plots/outliers/serie_386.png
   [ 8/20] ../../plots/outliers/serie_455.png
   [ 9/20] ../../plots/outliers/serie_597.png
   [10/20] ../../plots/outliers/serie_1496.png
   [11/20] ../../plots/outliers/serie_1630.png
   [12/20] ../../plots/outliers/serie_2225.png
   [13/20] ../../plots/outliers/serie_691.png
   [14/20] ../../plots/outliers/serie_1011.png
   [15/20] ../../plots/outliers/serie_1200.png
   [16/20] ../../plots/outliers/serie_723.png
   [17/20] ../../plots/outliers/serie_1070.png
   [18/20] ../../plots/outliers/serie_1097.png
   [19/20] ../../plots/outliers/serie_1210.png
   [20/20] ../../plots/outliers/serie_2264.png

✅ Plots salvos em

In [14]:
df_base

,week_date,time_series_id,units_sold,segment
33,2023-03-06,2,28,valid
34,2023-03-13,2,11,valid
35,2023-03-20,2,24,valid
36,2023-03-27,2,48,valid
37,2023-04-03,2,6,valid
...,...,...,...,...
203466,2024-07-29,2845,0,valid
203467,2024-08-05,2845,0,valid
203468,2024-08-12,2845,0,valid
203469,2024-08-19,2845,0,valid


In [ ]:
df_treated

,week_date,time_series_id,units_sold,is_outlier,rolling_baseline,upper_threshold,lower_threshold,units_sold_original,units_sold_capped,units_sold_removed,units_sold_interpolated
0,2023-01-09,1,2,False,NaN,NaN,NaN,2.0,2.0,2.0,2.0
1,2023-01-16,1,6,False,NaN,NaN,NaN,6.0,6.0,6.0,6.0
2,2023-01-23,1,0,False,NaN,NaN,NaN,0.0,0.0,0.0,0.0
3,2023-01-30,1,0,False,NaN,NaN,NaN,0.0,0.0,0.0,0.0
4,2023-02-06,1,0,False,NaN,NaN,NaN,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
203466,2024-07-29,2845,0,False,NaN,NaN,NaN,0.0,0.0,0.0,0.0
203467,2024-08-05,2845,0,False,NaN,NaN,NaN,0.0,0.0,0.0,0.0
203468,2024-08-12,2845,0,False,NaN,NaN,NaN,0.0,0.0,0.0,0.0
203469,2024-08-19,2845,0,False,NaN,NaN,NaN,0.0,0.0,0.0,0.0
